## 10-06-2026 ADVANCED OOP
The next layer of OOP: ways to build and extend objects (composition, dynamic extension), polymorphism and duck typing, the different kinds of variables and methods, decorators, dataclasses, and how to size a class.

Outline for today:
1. Encapsulation: composition, dynamic extension
2. Polymorphism and duck typing
3. Class variables vs instance variables
4. Instance methods
5. Class method
6. Static method
7. Decorators
8. dataclass and __post_init__
9. How long should a class be

### Encapsulation
Encapsulation is about how an object's structure and behaviour are put together and controlled. Two techniques covered today build on it: composition (assembling an object from other objects) and dynamic extension (adding to an object at runtime).

### Encapsulation: Composition
Inheritance is an "is a" relationship (a Student is a Person). Composition is a "has a" relationship (a Car has an Engine): instead of inheriting, the object holds another object inside it as an attribute and uses it. Composition is often preferred over inheritance because it is more flexible, I assemble a bigger object from smaller swappable parts.

In [1]:
# composition - an object HAS another object inside it (Car has an Engine)
class Engine:
    def __init__(self, hp): self.hp = hp
    def start(self): return f"Engine {self.hp} hp starting"

class Car:
    def __init__(self, model, hp):
        self.model = model
        self.engine = Engine(hp)        # Car HAS an Engine
    def start(self): return f"{self.model}: " + self.engine.start()

print(Car("Civic", 158).start())

Civic: Engine 158 hp starting


### Encapsulation: Dynamic Extension
Python is dynamic, so I can add attributes (and even methods) to an object after it has been created, without declaring them in the class. Adding an attribute affects only that one instance, not the whole class. It is handy but should be used carefully, since attributes that appear out of nowhere make code harder to follow.

To attach a function as a real method bound to one object, I use `types.MethodType`, which binds the function so it receives that object as `self`.

In [2]:
# add an attribute on the fly - only on this instance, not the class
class Student:
    def __init__(self, name):
        self.name = name

s = Student("Asha")
s.marks = 88                  # new attribute added at runtime, only on s
print(s.name, s.marks)

s2 = Student("Ravi")
print("does s2 have marks?", hasattr(s2, 'marks'))   # False, it was only added to s

Asha 88
does s2 have marks? False


In [3]:
# attach a function as a method bound to this object
import types
def shout(self):
    return self.name.upper()

s.shout = types.MethodType(shout, s)   # bind shout to s as a method
print(s.shout())

ASHA


### Polymorphism and Duck Typing
Polymorphism means "many forms": the same method name behaves differently depending on the object, so I can loop over different objects and call the same method, each running its own version.

Duck typing is Python's take on this: "if it walks like a duck and quacks like a duck, it is a duck." Python does not check an object's type, only whether it has the method being called. So a function works on any object that has the right method, with no shared base class needed. That is polymorphism without inheritance.

In [4]:
# polymorphism - same method name, different result per object
class Dog:
    def sound(self): return "Bark"
class Cat:
    def sound(self): return "Meow"
for a in [Dog(), Cat()]:
    print(a.sound())

Bark
Meow


In [5]:
# duck typing - the function does not check the type, only that .sound() exists
class Car:
    def sound(self): return "Vroom"
def make_sound(thing):
    return thing.sound()
print(make_sound(Dog()))
print(make_sound(Car()))      # unrelated class, still works

Bark
Vroom


### Class Variables vs Instance Variables
An instance variable is defined with `self.x` and each object has its own copy. A class variable is defined directly in the class body and is shared by every object of that class. A shared counter, or a shared constant like a school name, is a good use for a class variable.

In [6]:
# class variable is shared by all objects, instance variable is per object
class Student:
    school = "USF"        # class variable, shared by all students
    count = 0
    def __init__(self, name):
        self.name = name          # instance variable, per object
        Student.count += 1        # bump the shared counter

a = Student("Asha"); b = Student("Ravi")
print(a.school, b.school)         # both see the shared value
print("count:", Student.count)

USF USF
count: 2


### Instance Methods
The normal kind of method. Its first parameter is `self`, so it can read and change that object's own data. This is what I use most of the time. Below, `result()` uses this student's own marks to decide pass or fail, and `birthday()` changes the object's state.

In [7]:
# instance method - takes self, works on this object's own data
class Student:
    def __init__(self, name, marks, age):
        self.name = name
        self.marks = marks
        self.age = age
    def result(self):                 # reads this object's marks
        return "Pass" if self.marks >= 40 else "Fail"
    def birthday(self):               # changes this object's state
        self.age += 1
        return self.age

s = Student("Asha", 88, 20)
print(s.result())
print("age after birthday:", s.birthday())

Pass
age after birthday: 21


### Class Method
Marked with `@classmethod`. Its first parameter is `cls` (the class itself, not an instance), so it works on class level data rather than one object's data. A common use is a factory method: an alternative way to build an object. Here `from_string` builds a Student from a "name-marks" string, and `how_many` reports the shared count.

In [8]:
# @classmethod takes cls (the class). Good for factory methods and class data
class Student:
    count = 0
    def __init__(self, name, marks):
        self.name, self.marks = name, marks
        Student.count += 1
    @classmethod
    def from_string(cls, text):       # alternative constructor
        name, marks = text.split("-")
        return cls(name, int(marks))
    @classmethod
    def how_many(cls):
        return cls.count

s = Student.from_string("Meena-77")
print(s.name, s.marks)
print("count:", Student.how_many())

Meena 77
count: 1


### Static Method
Marked with `@staticmethod`. It takes neither `self` nor `cls`. It is just a plain function that lives inside the class because it is logically related to it. Use it for a helper that does not need any object or class data, like a validator.

Quick comparison: an instance method takes `self` and works on one object, a class method takes `cls` and works on the class, and a static method takes neither and is just a related helper.

In [9]:
# @staticmethod takes neither self nor cls, just a related helper
class Student:
    @staticmethod
    def is_valid_marks(marks):
        return 0 <= marks <= 100

print(Student.is_valid_marks(88))
print(Student.is_valid_marks(150))

True
False


### Decorators
A decorator is a function that wraps another function to add behaviour, without changing the original. The `@name` line above a function applies it. `@classmethod` and `@staticmethod` are built in decorators. A decorator takes a function, defines an inner wrapper that does extra work around the call, and returns that wrapper.

In [10]:
# decorator - wraps a function to add behaviour. @ applies it
def log_call(func):
    def wrapper(*args, **kwargs):
        print(f"calling {func.__name__}")
        result = func(*args, **kwargs)
        print(f"done {func.__name__}")
        return result
    return wrapper

@log_call                  # same as greet = log_call(greet)
def greet(name): return f"Hi {name}"
print(greet("Asha"))

calling greet
done greet
Hi Asha


### dataclass and __post_init__
Writing `__init__` and `__str__` by hand for a class that mostly holds data is repetitive. The `@dataclass` decorator generates `__init__`, `__repr__` and `__eq__` automatically from the field annotations. `__post_init__` runs right after the generated `__init__`, which is the place for validation or to compute a derived field from the inputs (here the grade is derived from the marks).

In [11]:
# @dataclass auto generates init, repr, eq. __post_init__ runs after
from dataclasses import dataclass

@dataclass
class Student:
    name: str
    marks: int
    grade: str = ""
    def __post_init__(self):                # runs after the generated __init__
        self.grade = "A" if self.marks >= 80 else "B" if self.marks >= 60 else "C"

s = Student("Asha", 88)
print(s)                                    # __repr__ is auto generated
print("grade:", s.grade)
print(Student("Asha", 88) == s)             # __eq__ is auto generated too

Student(name='Asha', marks=88, grade='A')
grade: A
True


### How Long Should a Class Be
There is no fixed line count. The guiding rule is the Single Responsibility Principle: a class should do one thing and have one reason to change.

Signs a class is too big: it handles several unrelated jobs (data plus file saving plus plotting plus emailing), its name has "and" or it is a catch all "Manager", or its methods do not share the same instance variables (low cohesion).

The fix is to split it: pull each responsibility into its own class and wire them together with composition. Smaller focused classes are easier to read, test and reuse, and methods should stay short too, each doing one clear step.

### Recap
- encapsulation builds objects up: composition (HAS a) and dynamic extension (add at runtime)
- polymorphism is one method name with many behaviours, duck typing needs only the method to exist
- class variable is shared, instance variable is per object
- instance method takes self, class method takes cls (factories), static method takes neither (helper)
- decorator wraps a function to add behaviour
- @dataclass removes boilerplate, __post_init__ for validation or derived fields
- keep a class to one responsibility, split big ones with composition